# Exercise 4: Autoregressive Image Modeling
## Submission date: 25\05\2026, 23:59.

Submitted by:

 **Student 1 Wael Zidan + [student ID removed]

 **Student 2 Tarek Diab + [student ID removed]

<font color='red'>Before submission, please make sure that all the plots and results are present as a colab notebook file</font>


## Configurations and initializations

This section loads libraries and configurations for various tasks for this course

In [ ]:
## Standard libraries
import os
import math
import numpy as np
import pandas as pd

## Imports for plotting
import matplotlib.pyplot as plt
plt.set_cmap('cividis')
%matplotlib inline
from IPython.display import set_matplotlib_formats
%config InlineBackend.figure_formats = ['svg', 'pdf']
from matplotlib.colors import to_rgb
import seaborn as sns

## Progress bar
from tqdm.notebook import tqdm

## PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim

# Torchvision
import torchvision
from torchvision.datasets import MNIST
from torchvision import transforms

# Path to the folder where the datasets are/should be downloaded (e.g. MNIST)
DATASET_PATH = "../data"

# Path to the folder where the models are saved
CHECKPOINT_PATH = "./"

# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Fetching the device that will be used throughout this notebook
device = torch.device("cpu") if not torch.cuda.is_available() else torch.device("cuda:0")
print("Using device", device)

In [ ]:
# Convert images from 0-1 to 0-255 (integers).
def discretize(sample):
    return (sample * 255).to(torch.long)

# Transformations applied on each image => only make them a tensor
transform = transforms.Compose([transforms.ToTensor(), discretize])

# Loading the training dataset. We need to split it into a training and validation part
train_dataset = MNIST(root=DATASET_PATH, train=True, transform=transform, download=True)

#pl.seed_everything(42)
train_set, val_set = torch.utils.data.random_split(train_dataset, [50000, 10000])

# Loading the test set
test_set = MNIST(root=DATASET_PATH, train=False, transform=transform, download=True)

# We define a set of data loaders that we can use for various purposes later.
train_loader = data.DataLoader(train_set, batch_size=128, shuffle=True, drop_last=True, pin_memory=True, num_workers=0)
val_loader = data.DataLoader(val_set, batch_size=128, shuffle=False, drop_last=False, num_workers=0)
test_loader = data.DataLoader(test_set, batch_size=128, shuffle=False, drop_last=False, num_workers=0)

print('Train size:', len(train_loader.dataset))
print('Validation size:', len(val_loader.dataset))
print('Test size:', len(test_loader.dataset))

In [ ]:
def show_imgs(imgs):
    num_imgs = imgs.shape[0] if isinstance(imgs, torch.Tensor) else len(imgs)
    nrow = min(num_imgs, 4)
    ncol = int(math.ceil(num_imgs/nrow))
    imgs = torchvision.utils.make_grid(imgs, nrow=nrow, pad_value=128)
    imgs = imgs.clamp(min=0, max=255)
    np_imgs = imgs.cpu().numpy()
    plt.figure(figsize=(1.5*nrow, 1.5*ncol))
    plt.imshow(np.transpose(np_imgs, (1,2,0)), interpolation='nearest')
    plt.axis('off')
    plt.show()

show_imgs([train_set[i][0] for i in range(8)])

## Define Masked Cnvolutions

In [ ]:
class MaskedConvolution(nn.Module):

    def __init__(self, c_in, c_out, mask, **kwargs):
        """
        Implements a convolution with mask applied on its weights.
        Inputs:
            c_in - Number of input channels
            c_out - Number of output channels
            mask - Tensor of shape [kernel_size_H, kernel_size_W] with 0s where
                   the convolution should be masked, and 1s otherwise.
            kwargs - Additional arguments for the convolution
        """
        super().__init__()
        # For simplicity: calculate padding automatically
        kernel_size = (mask.shape[0], mask.shape[1])

        # Actual convolution
        self.conv = nn.Conv2d(c_in, c_out, kernel_size, padding='same', **kwargs)

        # Mask as buffer => it is no parameter but still a tensor of the module
        # (must be moved with the devices)
        self.register_buffer('mask', mask[None,None].to(device))

    def forward(self, x):
        self.conv.weight.data *= self.mask # Ensures zero's at masked positions
        return self.conv(x)

In [ ]:
class VerticalStackConvolution(MaskedConvolution):

    def __init__(self, c_in, c_out, kernel_size=3, mask_center=False, **kwargs):
        # Mask out all pixels below. For efficiency, we could also reduce the kernel
        # size in height, but for simplicity, we stick with masking here.
        mask = torch.ones(kernel_size, kernel_size)
        mask[kernel_size//2+1:,:] = 0

        # For the very first convolution, we will also mask the center row
        if mask_center:
            mask[kernel_size//2,:] = 0

        super().__init__(c_in, c_out, mask, **kwargs)

class HorizontalStackConvolution(MaskedConvolution):

    def __init__(self, c_in, c_out, kernel_size=3, mask_center=False, **kwargs):
        # Mask out all pixels on the left. Note that our kernel has a size of 1
        # in height because we only look at the pixel in the same row.
        mask = torch.ones(1,kernel_size)
        mask[0,kernel_size//2+1:] = 0

        # For the very first convolution, we will also mask the center pixel
        if mask_center:
            mask[0,kernel_size//2] = 0

        super().__init__(c_in, c_out, mask, **kwargs)

## PixelCNN without Gating and dilations

Instead of the GatedMaskConv from the tutorial, we defined a MaskedConvLayer for the model.

In [ ]:
class MaskedConvLayer(nn.Module):

    def __init__(self, c_in, kernel_size=5, **kwargs):
        super().__init__()
        self.conv_vert  = VerticalStackConvolution(c_in, c_in,
                                                   kernel_size=kernel_size,
                                                   mask_center=False)
        self.conv_horiz = HorizontalStackConvolution(c_in, c_in,
                                                     kernel_size=kernel_size,
                                                     mask_center=False)
        self.conv_vert_to_horiz = nn.Conv2d(c_in, c_in, kernel_size=1)
        self.conv_horiz_1x1     = nn.Conv2d(c_in, c_in, kernel_size=1)

    def forward(self, v_stack, h_stack):
        v_feat = self.conv_vert(v_stack)
        v_out  = F.elu(v_feat)

        h_feat = self.conv_horiz(h_stack) + self.conv_vert_to_horiz(v_feat)
        h_feat = F.elu(h_feat)
        h_out  = self.conv_horiz_1x1(h_feat) + h_stack   # residual

        return v_out, h_out

In [ ]:
class PixelCNN(nn.Module):

    def __init__(self, c_in, c_hidden, num_layers=8, kernel_size=5, causal=True):
        """
        Non-gated, non-dilated PixelCNN.
        - causal=True  -> first layer masks the center pixel  (Q1)
        - causal=False -> first layer does NOT mask the center (Q2)
        """
        super().__init__()
        self.c_in = c_in
        self.c_hidden = c_hidden

        self.conv_vstack = VerticalStackConvolution(c_in, c_hidden,
                                                    kernel_size=kernel_size,
                                                    mask_center=causal)
        self.conv_hstack = HorizontalStackConvolution(c_in, c_hidden,
                                                      kernel_size=kernel_size,
                                                      mask_center=causal)

        self.conv_layers = nn.ModuleList([
            MaskedConvLayer(c_hidden, kernel_size=kernel_size)
            for _ in range(num_layers)
        ])

        self.conv_out = nn.Conv2d(c_hidden, c_in * 256, kernel_size=1)

    def forward(self, x):
        x = (x.float() / 255.0) * 2 - 1
        v_stack = self.conv_vstack(x)
        h_stack = self.conv_hstack(x)
        for layer in self.conv_layers:
            v_stack, h_stack = layer(v_stack, h_stack)
        out = self.conv_out(F.elu(h_stack))
        out = out.reshape(out.shape[0], 256, out.shape[1]//256,
                          out.shape[2], out.shape[3])
        return out

    def calc_nll(self, x):
        pred = self.forward(x)
        nll  = F.cross_entropy(pred, x, reduction='none')
        bpd  = nll.mean(dim=[1, 2, 3]) * np.log2(np.exp(1))
        return bpd.mean()

    @torch.no_grad()
    def sample(self, img_shape, img=None):
        if img is None:
            img = torch.zeros(img_shape, dtype=torch.long).to(device) - 1
        for h in tqdm(range(img_shape[2]), leave=False):
            for w in range(img_shape[3]):
                for c in range(img_shape[1]):
                    if (img[:, c, h, w] != -1).all().item():
                        continue
                    pred  = self.forward(img[:, :, :h+1, :])
                    probs = F.softmax(pred[:, :, c, h, w], dim=-1)
                    img[:, c, h, w] = torch.multinomial(probs, num_samples=1).squeeze(dim=-1)
        return img

Before you start training, you have to choose hyper-parameters for your model (e.g. number of layers).

We can check the full receptive field of the model on an MNIST image of size $28\times 28$ to help you decide which hyper-parameters to use:

In [ ]:
def show_center_recep_field(img, out):

    loss = out[0, :, out.shape[2]//2, out.shape[3]//2].sum()
    loss.backward(retain_graph=True)
    img_grads = img.grad.abs()
    img.grad.fill_(0)

    img_np = img_grads.squeeze().cpu().numpy()
    fig, ax = plt.subplots(1, 2, figsize=(6, 3))
    ax[0].imshow(img_np)
    ax[1].imshow(img_np > 0)

    show_center = (img_np[img_np.shape[0]//2, img_np.shape[1]//2] == 0)
    if show_center:
        center_pixel = np.zeros(img_np.shape + (4,))
        center_pixel[center_pixel.shape[0]//2, center_pixel.shape[1]//2, :] = \
            np.array([1.0, 0.0, 0.0, 1.0])
    for i in range(2):
        ax[i].axis('off')
        if show_center:
            ax[i].imshow(center_pixel)
    ax[0].set_title("Weighted receptive field")
    ax[1].set_title("Binary receptive field")
    plt.show()

In [ ]:
test_model = PixelCNN(c_in=1, c_hidden=128, num_layers=8,
                      kernel_size=5, causal=True).to('cpu')
inp = torch.zeros(1, 1, 28, 28).to('cpu')
inp.requires_grad_()
out = test_model(inp)
show_center_recep_field(inp, out.squeeze(dim=2))
del inp, out, test_model

## Model Training

In [ ]:
model = PixelCNN(c_in=1, c_hidden=128, num_layers=8,
                 kernel_size=5, causal=True)
print("Num params: {:,}".format(sum(p.numel() for p in model.parameters())))
model = torch.compile(model.to(device))

optimizer = optim.Adam(model.parameters(), lr=3e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.99)
epochs = 15

In [ ]:
# Training loop
train_bpd = []
val_bpd = []

for epoch in range(epochs):
    model.train()
    losses = []

    for imgs, _ in tqdm(train_loader, leave=False):
        imgs = imgs.to(device)
        optimizer.zero_grad()
        loss = model.calc_nll(imgs)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    train_bpd.append(np.mean(losses))
    scheduler.step()

    losses = []
    with torch.no_grad():
      for imgs, _ in tqdm(val_loader, leave=False):
          imgs = imgs.to(device)
          loss = model.calc_nll(imgs)
          losses.append(loss.item())

    val_bpd.append(np.mean(losses))

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_bpd[-1]:.4f}, Val Loss: {val_bpd[-1]:.4f}")

    #if (epoch + 1) % 5 == 0:
    #    sampled_images = model.sample(img_shape=(4, 1, 28, 28))
    #    show_imgs(sampled_images)

# Test loop
model.eval()
losses = []
for imgs, _ in tqdm(test_loader, leave=False):
    imgs = imgs.to(device)
    loss = model.calc_nll(imgs)
    losses.append(loss.item())

test_bpd = np.mean(losses)
print(f"Test Loss: {test_bpd:.4f}")
plt.figure(figsize=(7, 4))
plt.plot(range(1, epochs+1), train_bpd, 'o-', label='Train (BPD)')
plt.plot(range(1, epochs+1), val_bpd,   's-', label='Validation (BPD)')
plt.xlabel('Epoch')
plt.ylabel('Negative log-likelihood (bits / dim)')
plt.title('PixelCNN — training and validation NLL')
plt.legend()
plt.grid(True, alpha=0.4)
plt.show()
model.eval()
samples = model.sample(img_shape=(16, 1, 28, 28))
show_imgs(samples)

## Question 1

The model has two masked-convolution stacks: a vertical stack that covers the rows above the current pixel, and a horizontal stack that covers the same row to its left. Each layer of the model aplies one masked convolution per stack with an ELU activation and a residual connection, and the two stacks stay separate through all the layers. They are added together just before the final 1×1 convolution, which produces 256 logits per pixel and feeds into the softmax over intensity values.

The first layer uses mask_center=True so that the prediction at pixel i never sees input pixel i. This is what makes the model autoregressive in the sense of the chain rule. Deeper layers use mask_center=False, because their inputs are already a function of strictly earlier pixels, and masking the center again would only throw information away.

Gating and dilation are removed as the assignment requires. The gated activation iis replaced by ELU. The lecture does not say which non-gated activation to use, so this is a default choice on our side. The cross-connection from the vertical to the horizontal stack that appears in the tutorial is also removed, because the assignment describes the two stacks as separate components and asks for them to be combined only at the end.

The loss is the cross-entropy between the per-pixel softmax and the integer image, converted from nats to bits by dividing by ln 2. This is the bits-per-dimension metric used in the "Negative log-likelyhood" slide of the same lecture.
Hyperparameters: 8 layers, hidden width 128, kernel size 5, learning rate 3e-4, StepLR with gamma 0.99, 15 epochs.

## Question 2

In [ ]:
# Non-causal PixelCNN
model_nc = PixelCNN(c_in=1, c_hidden=128, num_layers=8,
                    kernel_size=5, causal=False)            # the only difference
print("Num params: {:,}".format(sum(p.numel() for p in model_nc.parameters())))
model_nc = torch.compile(model_nc.to(device))

optimizer_nc = optim.Adam(model_nc.parameters(), lr=3e-4)
scheduler_nc = optim.lr_scheduler.StepLR(optimizer_nc, step_size=1, gamma=0.99)

train_bpd_nc, val_bpd_nc = [], []
for epoch in range(epochs):
    model_nc.train()
    losses = []
    for imgs, _ in tqdm(train_loader, leave=False):
        imgs = imgs.to(device)
        optimizer_nc.zero_grad()
        loss = model_nc.calc_nll(imgs)
        loss.backward()
        optimizer_nc.step()
        losses.append(loss.item())
    train_bpd_nc.append(np.mean(losses))
    scheduler_nc.step()

    losses = []
    with torch.no_grad():
        for imgs, _ in tqdm(val_loader, leave=False):
            imgs = imgs.to(device)
            losses.append(model_nc.calc_nll(imgs).item())
    val_bpd_nc.append(np.mean(losses))

    print(f"Epoch {epoch+1}/{epochs}, "
          f"Train: {train_bpd_nc[-1]:.4f}, Val: {val_bpd_nc[-1]:.4f}")

# Test
model_nc.eval()
losses = []
for imgs, _ in tqdm(test_loader, leave=False):
    imgs = imgs.to(device)
    losses.append(model_nc.calc_nll(imgs).item())
print(f"Test Loss: {np.mean(losses):.4f}")
plt.figure(figsize=(7, 4))
plt.plot(range(1, epochs+1), train_bpd_nc, 'o-', label='train')
plt.plot(range(1, epochs+1), val_bpd_nc,   's-', label='validation')
plt.xlabel('epoch'); plt.ylabel('NLL (bits / dim)')
plt.title('Non-causal PixelCNN (NLL is not a valid likelihood here)')
plt.legend(); plt.grid(True, alpha=0.4); plt.show()

samples_nc = model_nc.sample(img_shape=(16, 1, 28, 28))
show_imgs(samples_nc)

We change one line in the first layer of the model: mask_center=True becomes mask_center=False. The brediction at pixel i is now allowed to depend on input pixel i directly.

The trainiing and validation NLL drops to nearly zero. The model has learned to copy each pixel from input to output, so on training data it can place almost all probability on the correct intensity value.

The chain-rule factorization only holds when each conditional depends on strictly earlier pixels. Once the model can read its own pixel from the input, the product is no longer a joint density, and the cross-entropy stops measuring what it usually measures.

The samples confirm the issue. During generation the unknown pixels are placeholders (we use −1 in the code), which is far outside the input distribution the model saw during training. The model has been trained to read pixel (h, w) directly from the input and reproduce it at sampling time there is no real value to read, so the generations look like noise rather than digits. The loss is low and the samples are broken at the same time, which is the point of the exersice.

## Question 3

In [ ]:
test_imgs = torch.stack([test_set[i][0] for i in range(4)]).to(device)
H, W = 14, 14

@torch.no_grad()
def model_logsoftmax(img):
    return F.log_softmax(model(img), dim=1)

(a) Easy. The first pixel has no predecessors in raster order, so p(x_{0,0}) is what the model outputs at position (0, 0). The masks at that position leave only zero-padding in view, so the output is independent of the input. One forward pass.

In [ ]:
@torch.no_grad()
def marginal_first():
    dummy = torch.zeros(1, 1, 28, 28, dtype=torch.long, device=device)
    return model_logsoftmax(dummy)[0, :, 0, 0, 0].exp().cpu()

plt.bar(range(256), marginal_first().numpy())
plt.title("(a) p(x_{0,0})"); plt.xlabel("intensity"); plt.show()

(b) Hard. The exact computation p(x_{14,14}) = Σ_{x_<14,14} p(x_<14,14) p(x_{14,14}|x_< 14,14) has 256^378 terms. Approximated by Monte Carlo: sample N=32 above-left prefixes from the model and average the model's conditional at (14, 14). Unbiased variance shrinks as 1/N. Expected shape: sharply peaked at 0, because the midle of MNIST is almost always background.

In [ ]:
@torch.no_grad()
def marginal_middle(N=32):
    p = torch.zeros(256, device=device)
    for _ in tqdm(range(N), desc="(b) MC marginal"):
        # Sample the prefix x_<14,14 row by row
        img = torch.zeros(1,1,28,28, dtype=torch.long, device=device) - 1
        for h in range(H+1):
            wmax = W if h == H else 28
            for w in range(wmax):
                logp = model_logsoftmax(img[:,:,:h+1,:])[0,:,0,h,w]
                img[0,0,h,w] = torch.multinomial(logp.exp(), 1)
        # Model's conditional at the middle pixel
        logp_mid = model_logsoftmax(img[:,:,:H+1,:])[0,:,0,H,W]
        p += logp_mid.exp()
    return (p / N).cpu()

p_b = marginal_middle(N=32)
plt.figure(figsize=(7,3))
plt.bar(range(256), p_b.numpy()); plt.title("(b) p(x_{14,14})  — MC over 32 prefixes")
plt.xlabel("pixel value"); plt.ylabel("probability"); plt.show()

c) Easy. The conditioning set, "all pixels above and to the left of (14, 14)", is exactly the raster-order prefix x_< 14,14. This is the conditional the PixelCNN parametrizes by construction, so one fowrard pass on the conditioning image returns it directly.

In [ ]:
# section (c)
@torch.no_grad()
def cond_above_left(img):
    return model_logsoftmax(img.unsqueeze(0))[0, :, 0, H, W].exp().cpu()

fig, axes = plt.subplots(2, 4, figsize=(12, 5))
for i in range(4):
    axes[0, i].imshow(test_imgs[i, 0].cpu(), cmap='gray'); axes[0, i].axis('off')
    axes[1, i].bar(range(256), cond_above_left(test_imgs[i]).numpy())
plt.suptitle("(c) p(x_{14,14} | above-left)")
plt.tight_layout(); plt.show()

(d) Hard. The conditioning set is the suffix in raster order, and the model parametrizes conditionals on the prefix, not the suffix. By Bayes,
p(x_{14,14}=v | x_BR) ∝ E_{x_AL ~ p}[ p(x_{14,14}=v | x_AL) · p(x_BR | x_AL, x_{14,14}=v) ],
 We sample N = 4 above-left prefixes from the model, build 256 candidite completions per prefix, and score each under the model. The 256 scores are accumulated in log-space across prefixes and normalized. The same notes also worn that importance sampling degenerates in high dimensions: the bottom-right has about 392 pixels, so each weight is a product of about 392 model probabilities, and a few prefixes will dominate. The result with N = 4 is therefore rough, as the assignment permits. The principled alternative would be Gibbs sampling using the conditional from part (e), at much higher cost.

In [ ]:
@torch.no_grad()
def cond_below_right(img_target, N=8):
    img_target = img_target.to(device)
    log_accum = torch.full((256,), -float('inf'), device=device)

    for _ in tqdm(range(N), desc="(d) IS below-right"):
        # 1. Sample x_AL (everything before pixel (H,W)) from the prior
        img = torch.zeros(1,1,28,28, dtype=torch.long, device=device) - 1
        for h in range(H+1):
            wmax = W if h == H else 28
            for w in range(wmax):
                logp = model_logsoftmax(img[:,:,:h+1,:])[0,:,0,h,w]
                img[0,0,h,w] = torch.multinomial(logp.exp(), 1)

        # 2. log p(x_{H,W}=v | x_AL)  for v = 0..255
        logp_mid = model_logsoftmax(img[:,:,:H+1,:])[0,:,0,H,W]   # [256]

        # 3. Build batch of 256 candidate images (one per v) with the bottom-right
        #    filled from img_target, and score them.
        batch = img.expand(256,1,28,28).clone()
        for v in range(256): batch[v,0,H,W] = v
        batch[:,0,H,W+1:] = img_target[0,H,W+1:]
        batch[:,0,H+1:,:] = img_target[0,H+1:,:]

        logp_full = model_logsoftmax(batch)
        # Sum log p over the BR positions (same row after W, and all rows below)
        idx = torch.arange(256, device=device)
        logp_br = torch.zeros(256, device=device)
        for w in range(W+1, 28):
            logp_br += logp_full[idx, batch[:,0,H,w], 0, H, w]
        for h in range(H+1, 28):
            for w in range(28):
                logp_br += logp_full[idx, batch[:,0,h,w], 0, h, w]

        # accumulate in log-space (logsumexp for numerical stability)
        log_accum = torch.logaddexp(log_accum, logp_mid + logp_br)

    log_p = log_accum - torch.logsumexp(log_accum, dim=0)
    return log_p.exp().cpu()

fig, axes = plt.subplots(2, 3, figsize=(10,5))
for i in range(3):
    axes[0,i].imshow(test_imgs[i,0].cpu(), cmap='gray'); axes[0,i].axis('off')
    axes[0,i].set_title(f"conditioning img {i}")
    p = cond_below_right(test_imgs[i], N=8)
    axes[1,i].bar(range(256), p.numpy()); axes[1,i].set_title("p(x_{14,14}|BR)")
plt.suptitle("(d) Conditional on below-right pixels  (importance sampling, N=8)")
plt.tight_layout(); plt.show()

(e) Easy in practice. By Bayes, p(x_{14,14}=v | rest) ∝ p(image with x_{14,14} set to v), and the full-image likelihood is computable in one batched forward pass over 256 candidate images. We then normalize over v. This conditional should be the sharpest of (c), (d), (e), because both the past and the future of the midle pixel are observed.

In [ ]:
@torch.no_grad()
def cond_all_others(img_target):
    img_target = img_target.to(device)
    batch = img_target.unsqueeze(0).expand(256, 1, 28, 28).clone()
    for v in range(256):
        batch[v, 0, H, W] = v

    logp = model_logsoftmax(batch)
    idx  = torch.arange(256, device=device)
    total = torch.zeros(256, device=device)
    for h in range(28):
        for w in range(28):
            total += logp[idx, batch[:, 0, h, w], 0, h, w]
    return (total - torch.logsumexp(total, dim=0)).exp().cpu()

fig, axes = plt.subplots(2, 4, figsize=(12, 5))
for i in range(4):
    axes[0, i].imshow(test_imgs[i, 0].cpu(), cmap='gray'); axes[0, i].axis('off')
    axes[1, i].bar(range(256), cond_all_others(test_imgs[i]).numpy())
plt.suptitle("(e) p(x_{14,14} | rest)")
plt.tight_layout(); plt.show()

(f) Easy. The top half is a raster-order prefix, so ancestral sampling fills in the bottom half pixel by pixel using p(x_i | x_< i) from the model. The existing model.sample already supports this: any pixel left at the placeholder -1 is resampled, the rest stay fixed. Running it twice on the same top halves gives different bottoms, which is what we want.

In [ ]:
@torch.no_grad()
def bottomGiven_top(top_imgs):
    seed = top_imgs.clone().to(device).long()
    seed[:, :, 14:, :] = -1
    return model.sample(img_shape=seed.shape, img=seed)

show_imgs(bottomGiven_top(test_imgs[:4]))

(g) Hard. We want to sample against the raster order. By Bayes, p(top | bottom) ∝ p(top) · p(bottom | top). We approximate with importance sampling: draw K = 64 top halves from the prior using batched ancestral sampling for rows 0..13, then score each candidate by the model's likelihood of the observed bottom half under it, and resample according to those weights. As in part (d), the bottom half is about 392 pixels, so the weights are products of about 392 model probabilities and degenerate in the way the MCMC notes describe. K = 64 gives a usable but rough estimate; a faithful method would be Gibbs sampling over the 392 top pixels using the conditional from part (e), with the same expense argument as in (d).

In [ ]:
@torch.no_grad()
def topGiven_bottom(bottom_img, K=64, keep=4):
    """
    Importance sampling:
      1. draw K full images by sampling top from the prior and ignoring bottom
      2. weight each by p(observed_bottom | sampled_top) under the model
      3. resample `keep` indices according to weights
    """
    bottom_img = bottom_img.to(device).long()
    # 1. Sample K candidate full images from the unconditional prior
    cands = model.sample(img_shape=(K,1,28,28))

    # 2. For each candidate, replace bottom half with the OBSERVED bottom and
    #    score the bottom under the model.
    scored = cands.clone()
    scored[:, :, 14:, :] = bottom_img[0, 14:, :]
    logp = model_logsoftmax(scored)                      # [K,256,1,28,28]
    idx = torch.arange(K, device=device)
    log_w = torch.zeros(K, device=device)
    for h in range(14, 28):
        for w in range(28):
            log_w += logp[idx, scored[:,0,h,w], 0, h, w]

    # 3. Resample
    w = torch.softmax(log_w, dim=0)
    pick = torch.multinomial(w, keep, replacement=True)
    out = cands[pick].clone()
    out[:, :, 14:, :] = bottom_img[0, 14:, :]            # paste true bottom for display
    return out

# Try with a few different bottom-halves
for i in range(3):
    print(f"Conditioning on bottom half of test image {i}")
    samples_g = topGiven_bottom(test_imgs[i], K=64, keep=4)
    show_imgs(samples_g)